In [ ]:
!pip install dlt[bigquery]

In [ ]:
import itertools, os
from tqdm import tqdm_notebook
year = ['2019','2020']
months = list(map(lambda x: '0'+str(x) if x<10 else str(x), list(range(1,13))))
year_months = set(itertools.product(months, year))

In [ ]:
URLS = []
for month, year in year_months:
    YELLOW_URL = f"https://github.com/DataTalksClub/nyc-tlc-data/releases/download/yellow/yellow_tripdata_{year}-{month}.csv.gz"
    GREEN_URL = f"https://github.com/DataTalksClub/nyc-tlc-data/releases/download/green/green_tripdata_{year}-{month}.csv.gz"
    URLS.extend([YELLOW_URL, GREEN_URL])

In [ ]:
os.makedirs('data/yellow')
os.makedirs('data/green')

In [ ]:
for URL in tqdm_notebook(URLS):
    dirpath = ['data']
    dirpath.extend(URL.split('/')[-2:])
    dirpath = '/'.join(dirpath)
    os.system(f"wget {URL} -O {dirpath}")


In [ ]:
import dlt
from dlt.sources.filesystem import filesystem, read_csv

files = filesystem(bucket_url="data/green", file_glob="*.csv.gz")
files.apply_hints(incremental=dlt.sources.incremental("modification_date"))
reader = (files | read_csv()).with_name("green_tripdata")
pipeline = dlt.pipeline(pipeline_name="green_taxi_datapipe", dataset_name="trips_newdata", destination="bigquery")

info = pipeline.run(reader)
print(info)

In [ ]:
import dlt
from dlt.sources.filesystem import filesystem, read_csv

files = filesystem(bucket_url="data/yellow", file_glob="*.csv.gz")
files.apply_hints(incremental=dlt.sources.incremental("modification_date"))
reader = (files | read_csv()).with_name("yellow_tripdata")
pipeline = dlt.pipeline(pipeline_name="yellow_taxi_datapipe", dataset_name="trips_newdata", destination="bigquery")

info = pipeline.run(reader)
print(info)

In [ ]:
year_months

In [ ]:
URL_FHV = []
for month, year in year_months:
    FHV_URL = f"https://github.com/DataTalksClub/nyc-tlc-data/releases/download/fhv/fhv_tripdata_{year}-{month}.csv.gz"
    URL_FHV.append(FHV_URL)

os.makedirs('data/fhv')

for URL in tqdm_notebook(URL_FHV):
    dirpath = ['data']
    dirpath.extend(URL.split('/')[-2:])
    dirpath = '/'.join(dirpath)
    os.system(f"wget {URL} -O {dirpath}")
